In [1]:
# Βιβλιοθήκες για τη διαχείριση αρχείων, δεδομένων και API requests
import os
import html
import requests
import pandas as pd

from dotenv import load_dotenv
from datetime import datetime, timezone
from pathlib import Path
from IPython.display import display


# Φόρτωση των μεταβλητών που είναι αποθηκευμένες στο αρχείο .env
load_dotenv()

# Ανάκτηση του YouTube API key χωρίς να εμφανίζεται μέσα στον κώδικα
API_KEY = os.getenv("YOUTUBE_API_KEY")

# Διακοπή της εκτέλεσης αν το API key δεν βρεθεί
if not API_KEY:
    raise ValueError("Δεν βρέθηκε το YOUTUBE_API_KEY στο αρχείο .env")

# Δημιουργία session για την επαναχρησιμοποίηση της σύνδεσης με το API
session = requests.Session()

print("Το API key φορτώθηκε επιτυχώς από το .env.")

Το API key φορτώθηκε επιτυχώς από το .env.


In [2]:
# Τα 4 θέματα και οι αντίστοιχοι όροι αναζήτησης στο YouTube
TOPICS = {
    "Football": "football",
    "Climate Change": "climate change",
    "Video Games": "video games",
    "Artificial Intelligence": "artificial intelligence"
}

# Αριθμός σχολίων που θέλουμε να συλλέξουμε για κάθε θέμα
TARGET_PER_TOPIC = 250

# Μέγιστος αριθμός βίντεο που θα αναζητηθούν για κάθε θέμα
VIDEOS_PER_TOPIC = 40

# Διευθύνσεις του YouTube API για αναζήτηση βίντεο και συλλογή σχολίων
SEARCH_URL = "https://www.googleapis.com/youtube/v3/search"
COMMENTS_URL = "https://www.googleapis.com/youtube/v3/commentThreads"

# Εμφάνιση μιας σύντομης περίληψης των ρυθμίσεων
print(f"Topics: {len(TOPICS)}")
print(f"Στόχος ανά topic: {TARGET_PER_TOPIC}")
print(f"Συνολικός στόχος: {TARGET_PER_TOPIC * len(TOPICS)} σχόλια")

Topics: 4
Στόχος ανά topic: 250
Συνολικός στόχος: 1000 σχόλια


In [3]:
# Λίστα στην οποία θα αποθηκευτούν τα στοιχεία των βίντεο
videos = []

# Αναζήτηση βίντεο ξεχωριστά για κάθε topic
for topic, search_query in TOPICS.items():

    # Παράμετροι που στέλνονται στο YouTube Search API
    params = {
        "part": "snippet",
        "q": search_query,
        "type": "video",
        "maxResults": VIDEOS_PER_TOPIC,
        "order": "relevance",
        "relevanceLanguage": "en",
        "safeSearch": "moderate",
        "key": API_KEY
    }

    # Αποστολή του αιτήματος στο API
    response = session.get(
        SEARCH_URL,
        params=params,
        timeout=30
    )

    # Διακοπή της διαδικασίας αν το API επιστρέψει σφάλμα
    if response.status_code != 200:
        error = response.json().get("error", {})

        raise RuntimeError(
            f"YouTube Search API error {response.status_code}: "
            f"{error.get('message', 'Unknown error')}"
        )

    # Μετατροπή της απάντησης του API σε Python dictionary
    data = response.json()

    # Καταγραφή των βασικών πληροφοριών κάθε βίντεο
    for search_rank, item in enumerate(
        data.get("items", []),
        start=1
    ):
        snippet = item["snippet"]
        video_id = item["id"]["videoId"]

        videos.append({
            "topic": topic,
            "search_query": search_query,
            "search_rank": search_rank,
            "video_id": video_id,
            "video_title": html.unescape(snippet["title"]),
            "video_channel": html.unescape(
                snippet["channelTitle"]
            ),
            "video_published_at": snippet["publishedAt"],
            "video_url": (
                f"https://www.youtube.com/watch?v={video_id}"
            )
        })


# Μετατροπή της λίστας σε DataFrame και αφαίρεση διπλότυπων βίντεο
videos_df = (
    pd.DataFrame(videos)
    .drop_duplicates(subset=["topic", "video_id"])
    .reset_index(drop=True)
)

# Έλεγχος του αριθμού των βίντεο που βρέθηκαν για κάθε topic
video_summary = (
    videos_df
    .groupby("topic")
    .agg(videos_found=("video_id", "nunique"))
    .reindex(TOPICS.keys())
)

display(video_summary)

print(f"Συνολικά videos: {len(videos_df)}")

,videos_found
topic,
Football,40
Climate Change,40
Video Games,40
Artificial Intelligence,40


Συνολικά videos: 160


In [4]:
def get_api_error(response):
    """Επιστρέφει τον λόγο και το μήνυμα ενός API error."""

    try:
        # Ανάκτηση των πληροφοριών του σφάλματος από την απάντηση
        error = response.json().get("error", {})
        errors = error.get("errors", [])

        reason = (
            errors[0].get("reason", "")
            if errors
            else ""
        )

        message = error.get(
            "message",
            "Unknown YouTube API error"
        )

    # Χρησιμοποιείται όταν η απάντηση δεν μπορεί να μετατραπεί σε JSON
    except ValueError:
        reason = ""
        message = response.text

    return reason, message


def fetch_comment_page(video_row, page_token=None):
    """Συλλέγει μία σελίδα από top-level σχόλια ενός βίντεο."""

    # Παράμετροι για τη συλλογή έως 100 σχολίων
    params = {
        "part": "snippet",
        "videoId": video_row["video_id"],
        "maxResults": 100,
        "order": "time",
        "textFormat": "plainText",
        "key": API_KEY
    }

    # Χρησιμοποιείται όταν ζητάμε την επόμενη σελίδα σχολίων
    if page_token:
        params["pageToken"] = page_token

    # Αποστολή αιτήματος στο YouTube Comments API
    response = session.get(
        COMMENTS_URL,
        params=params,
        timeout=30
    )

    # Διαχείριση πιθανών σφαλμάτων του API
    if response.status_code != 200:
        reason, message = get_api_error(response)

        # Περιπτώσεις στις οποίες το βίντεο παραλείπεται
        unavailable_reasons = {
            "commentsDisabled",
            "videoNotFound",
            "forbidden"
        }

        if reason in unavailable_reasons:
            print(
                f"Παράλειψη video {video_row['video_id']}: "
                f"{reason}"
            )
            return [], None

        # Τα υπόλοιπα σφάλματα σταματούν την εκτέλεση
        raise RuntimeError(
            f"YouTube Comments API error "
            f"{response.status_code}: {message}"
        )

    data = response.json()

    # Καταγραφή της ημερομηνίας και ώρας συλλογής
    collected_at = datetime.now(timezone.utc).isoformat()

    records = []

    # Επεξεργασία των σχολίων που επέστρεψε το API
    for item in data.get("items", []):
        thread_snippet = item["snippet"]
        top_comment = thread_snippet["topLevelComment"]
        comment_snippet = top_comment["snippet"]

        comment_id = top_comment["id"]

        # Το author channel ID μπορεί να μην υπάρχει σε όλα τα σχόλια
        author_channel = (
            comment_snippet
            .get("authorChannelId", {})
            .get("value")
        )

        # Προτιμάται το αρχικό κείμενο και χρησιμοποιείται εναλλακτικά το display text
        text = (
            comment_snippet.get("textOriginal")
            or comment_snippet.get("textDisplay", "")
        )

        # Αποθήκευση των διαθέσιμων στοιχείων του σχολίου
        records.append({
            "comment_id": comment_id,
            "author_name": html.unescape(
                comment_snippet.get(
                    "authorDisplayName",
                    ""
                )
            ),
            "author_channel_id": author_channel,
            "topic": video_row["topic"],
            "search_query": video_row["search_query"],
            "text": text,
            "published_at": comment_snippet.get(
                "publishedAt"
            ),
            "updated_at": comment_snippet.get(
                "updatedAt"
            ),
            "like_count": comment_snippet.get(
                "likeCount",
                0
            ),
            "reply_count": thread_snippet.get(
                "totalReplyCount",
                0
            ),
            "video_id": video_row["video_id"],
            "video_title": video_row["video_title"],
            "video_channel": video_row["video_channel"],
            "search_rank": video_row["search_rank"],
            "comment_url": (
                "https://www.youtube.com/watch?"
                f"v={video_row['video_id']}"
                f"&lc={comment_id}"
            ),
            "collected_at": collected_at
        })

    # Επιστρέφονται τα σχόλια και το token της επόμενης σελίδας
    return records, data.get("nextPageToken")

In [5]:
# Εδώ θα αποθηκευτεί το τελικό DataFrame κάθε topic
topic_datasets = []

# Χρησιμοποιείται για την αποφυγή διπλότυπων σχολίων μεταξύ των topics
global_comment_ids = set()

for topic in TOPICS:

    print(f"\nΣυλλογή topic: {topic}")

    # Επιλογή και ταξινόμηση των βίντεο του συγκεκριμένου topic
    topic_videos = (
        videos_df[videos_df["topic"] == topic]
        .sort_values("search_rank")
        .reset_index(drop=True)
    )

    # Προσωρινή αποθήκευση σχολίων και IDs για το τρέχον topic
    topic_pool = []
    topic_comment_ids = set()

    # Αποθήκευση του επόμενου page token για κάθε βίντεο
    next_page_tokens = {}

    # Συλλογή της πρώτης σελίδας σχολίων από κάθε βίντεο
    for _, video_row in topic_videos.iterrows():

        batch, next_token = fetch_comment_page(
            video_row
        )

        # Προσθήκη μόνο μοναδικών σχολίων
        for record in batch:
            comment_id = record["comment_id"]

            if (
                comment_id not in topic_comment_ids
                and comment_id not in global_comment_ids
            ):
                topic_pool.append(record)
                topic_comment_ids.add(comment_id)

        # Αποθήκευση του token αν υπάρχει επόμενη σελίδα
        if next_token:
            next_page_tokens[
                video_row["video_id"]
            ] = next_token

    # Συλλογή επιπλέον σελίδων μέχρι να φτάσουμε τα 250 σχόλια
    while (
        len(topic_pool) < TARGET_PER_TOPIC
        and next_page_tokens
    ):
        comments_before = len(topic_pool)

        for _, video_row in topic_videos.iterrows():
            video_id = video_row["video_id"]

            # Παράλειψη βίντεο που δεν έχει άλλη σελίδα σχολίων
            if video_id not in next_page_tokens:
                continue

            batch, next_token = fetch_comment_page(
                video_row,
                next_page_tokens[video_id]
            )

            # Προσθήκη μόνο σχολίων που δεν έχουν ήδη συλλεχθεί
            for record in batch:
                comment_id = record["comment_id"]

                if (
                    comment_id not in topic_comment_ids
                    and comment_id not in global_comment_ids
                ):
                    topic_pool.append(record)
                    topic_comment_ids.add(comment_id)

            # Ενημέρωση ή αφαίρεση του page token του βίντεο
            if next_token:
                next_page_tokens[video_id] = next_token
            else:
                del next_page_tokens[video_id]

            # Σταματάμε μόλις καλυφθεί ο στόχος
            if len(topic_pool) >= TARGET_PER_TOPIC:
                break

        # Προστασία από συνεχή επανάληψη αν δεν βρεθούν νέα σχόλια
        if len(topic_pool) == comments_before:
            break

    # Διακοπή αν δεν βρέθηκαν αρκετά σχόλια για το topic
    if len(topic_pool) < TARGET_PER_TOPIC:
        raise RuntimeError(
            f"Βρέθηκαν μόνο {len(topic_pool)} σχόλια "
            f"για το topic '{topic}'."
        )

    topic_df = pd.DataFrame(topic_pool)

    # Αρίθμηση των σχολίων μέσα σε κάθε βίντεο
    topic_df["position_in_video"] = (
        topic_df
        .groupby("video_id")
        .cumcount()
    )

    # Επιλογή 250 σχολίων με όσο γίνεται καλύτερη κατανομή μεταξύ των βίντεο
    topic_df = (
        topic_df
        .sort_values(
            [
                "position_in_video",
                "search_rank",
                "published_at"
            ],
            ascending=[True, True, False]
        )
        .head(TARGET_PER_TOPIC)
        .drop(columns="position_in_video")
        .reset_index(drop=True)
    )

    # Καταγραφή των τελικών comment IDs ώστε να μην επαναχρησιμοποιηθούν
    global_comment_ids.update(
        topic_df["comment_id"].tolist()
    )

    topic_datasets.append(topic_df)

    print(
        f"Ολοκληρώθηκε: {len(topic_df)} σχόλια "
        f"από {topic_df['video_id'].nunique()} videos"
    )


# Ένωση των τεσσάρων topics σε ένα ενιαίο DataFrame
comments_df = pd.concat(
    topic_datasets,
    ignore_index=True
)


Συλλογή topic: Football
Παράλειψη video Rig6iPQ2oWc: commentsDisabled
Παράλειψη video 8aK7bHybAcw: commentsDisabled
Παράλειψη video e_-DZUNK6oc: commentsDisabled
Ολοκληρώθηκε: 250 σχόλια από 35 videos

Συλλογή topic: Climate Change
Παράλειψη video SDRxfuEvqGg: commentsDisabled
Παράλειψη video k3yL_1L85Mk: commentsDisabled
Ολοκληρώθηκε: 250 σχόλια από 38 videos

Συλλογή topic: Video Games
Παράλειψη video Uen-aD8kTMY: commentsDisabled
Παράλειψη video UzbZfQiDouQ: commentsDisabled
Ολοκληρώθηκε: 250 σχόλια από 38 videos

Συλλογή topic: Artificial Intelligence
Παράλειψη video _19pRsZRiz4: commentsDisabled
Παράλειψη video JcXKbUIebrU: commentsDisabled
Ολοκληρώθηκε: 250 σχόλια από 38 videos


In [6]:
# Τελικές στήλες και σειρά εμφάνισής τους στο dataset
FINAL_COLUMNS = [
    "comment_id",
    "author_name",
    "author_channel_id",
    "topic",
    "search_query",
    "text",
    "published_at",
    "updated_at",
    "like_count",
    "reply_count",
    "video_id",
    "video_title",
    "video_channel",
    "search_rank",
    "comment_url",
    "collected_at"
]

comments_df = comments_df[FINAL_COLUMNS]

# Περίληψη των σχολίων και των βίντεο ανά topic
summary = (
    comments_df
    .groupby("topic")
    .agg(
        comments=("comment_id", "count"),
        unique_comments=("comment_id", "nunique"),
        videos_used=("video_id", "nunique")
    )
    .reindex(TOPICS.keys())
)

display(summary)

# Έλεγχος για διπλότυπα comment IDs
duplicate_ids = comments_df["comment_id"].duplicated().sum()

print(f"Συνολικά σχόλια: {len(comments_df)}")
print(f"Διπλότυπα comment IDs: {duplicate_ids}")

# Αυτόματοι έλεγχοι για την επιβεβαίωση του τελικού dataset
assert len(comments_df) == 1000
assert comments_df["comment_id"].is_unique
assert (
    comments_df.groupby("topic")
    .size()
    .eq(TARGET_PER_TOPIC)
    .all()
)

# Δημιουργία του φακέλου data/raw 
output_folder = Path("../data/raw")
output_folder.mkdir(parents=True, exist_ok=True)

# Όνομα και διαδρομή του τελικού raw αρχείου
output_file = output_folder / "youtube_comments_raw.csv"

# Αποθήκευση σε CSV 
comments_df.to_csv(
    output_file,
    index=False,
    encoding="utf-8-sig"
)

print(f"\nΑποθηκεύτηκε επιτυχώς στο: {output_file}")

# Προεπισκόπηση των πρώτων πέντε γραμμών
display(comments_df.head())

,comments,unique_comments,videos_used
topic,,,
Football,250,250,35
Climate Change,250,250,38
Video Games,250,250,38
Artificial Intelligence,250,250,38


Συνολικά σχόλια: 1000
Διπλότυπα comment IDs: 0

Αποθηκεύτηκε επιτυχώς στο: ../data/raw/youtube_comments_raw.csv


,comment_id,author_name,author_channel_id,topic,search_query,text,published_at,updated_at,like_count,reply_count,video_id,video_title,video_channel,search_rank,comment_url,collected_at
0,UgwwPpIztdWx-prfyQF4AaABAg,@YusuphaJaiteh-q2e,UCeY0u_3Qm1dLS7zMi8TNzew,Football,football,I see only one Brighton’s goals,2026-08-05T16:33:37Z,2026-08-05T16:33:37Z,0,0,y4kH9qrV0Ec,1 AMAZING Premier League Goal Scored From Ever...,Premier League,2,https://www.youtube.com/watch?v=y4kH9qrV0Ec&lc...,2026-08-05T20:00:15.013925+00:00
1,Ugxr7nNrhPTxjsgjxRR4AaABAg,@ChuffFootyReacts,UCh9NvHKmFu-wwz5-OTrS7sQ,Football,football,Missed the OG Pop drop? 🔥Extended for a limit...,2026-08-03T16:43:50Z,2026-08-03T16:43:50Z,14,3,4-8ba3lV0IA,Celebrities Playing Football,Chuff Footy Reacts,3,https://www.youtube.com/watch?v=4-8ba3lV0IA&lc...,2026-08-05T20:00:15.300985+00:00
2,UgxRfPnY5QXo_l4MT914AaABAg,@jdncourtneymiller6088,UCgOyq_cn5Hm0k-VUbuEETsw,Football,football,WDE,2026-08-05T19:51:23Z,2026-08-05T19:51:23Z,0,0,eM5_UjrTrA0,Reacting To Auburn Football's First Fall Practice,Locked On Auburn,4,https://www.youtube.com/watch?v=eM5_UjrTrA0&lc...,2026-08-05T20:00:15.418073+00:00
3,UgyvEH54yKWd3ST-jbV4AaABAg,@saritatripathirichhariya5134,UCvmOlKfbsn5x3DJllY9LirQ,Football,football,Hi,2026-08-02T07:57:51Z,2026-08-02T07:57:51Z,0,0,pcAPOSv0OjA,SPAIN 1-0 ARGENTINA | FIFA WORLD CUP 2026 FINA...,BOBOLA TV,5,https://www.youtube.com/watch?v=pcAPOSv0OjA&lc...,2026-08-05T20:00:15.706850+00:00
4,UgzCOplNmfHqydy_RrB4AaABAg,@baconboiii-e2c,UCpBLnUbMJt1Gs0aVuMYNRCA,Football,football,who thinks 2022 wc was better than this one pr...,2026-08-05T19:32:46Z,2026-08-05T19:32:46Z,1,0,bbx15WopTLI,We took Soccer Football to America’s Backyard ...,adidas,6,https://www.youtube.com/watch?v=bbx15WopTLI&lc...,2026-08-05T20:00:15.872678+00:00
